# Joint Distribution Analysis — PCAFactor Model (60 Assets, 3 Classes)

Compares the joint distribution of generated vs real returns across the three asset classes:
- **Bonds** (20 assets)
- **Commodities** (20 assets)
- **Stocks** (20 assets)

Data: test period (2024–2025, 251 days), 200 generated paths averaged.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

REPO_ROOT  = Path("..")
PCA_ROOT   = REPO_ROOT / "PCAFactor_Model"
DATA_DIR   = PCA_ROOT / "data"

# ── Load metadata ─────────────────────────────────────────────────────────────
meta        = json.loads((DATA_DIR / "meta.json").read_text())
asset_list  = meta["asset_list"]   # 60 names
class_idx   = meta["class_indices"]  # {bonds: [0..19], commodities: [20..39], stocks: [40..59]}

CLASSES = {
    "Bonds":       class_idx["bonds"],
    "Commodities": class_idx["commodities"],
    "Stocks":      class_idx["stocks"],
}
CLASS_COLORS = {"Bonds": "steelblue", "Commodities": "darkorange", "Stocks": "seagreen"}

# ── Load generated returns (mean across 200 paths) ────────────────────────────
gen_df = pd.read_csv(DATA_DIR / "generated_returns_test.csv", index_col="Date", parse_dates=True)

# ── Load real returns and align to test period ────────────────────────────────
real_df = pd.read_csv(DATA_DIR / "all_assets_log_returns.csv", index_col="Date", parse_dates=True)
real_test = real_df.loc[gen_df.index[0]:gen_df.index[-1]]

# ── Load all paths for uncertainty bands ─────────────────────────────────────
all_paths = np.load(DATA_DIR / "generated_paths_test.npy")  # (200, 251, 60)

print(f"Generated test: {gen_df.shape}")
print(f"Real test:      {real_test.shape}")
print(f"All paths:      {all_paths.shape}")
print(f"Test period:    {gen_df.index[0].date()} → {gen_df.index[-1].date()}")

## 1. Pooled Return Distribution — Per Asset Class

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
clip = (-0.08, 0.08)

for ax, (cls, indices) in zip(axes, CLASSES.items()):
    cols = [asset_list[i] for i in indices]

    real_vals = real_test[cols].values.flatten()
    real_vals = real_vals[~np.isnan(real_vals)]
    real_vals = real_vals[(real_vals >= clip[0]) & (real_vals <= clip[1])]

    gen_vals  = gen_df[cols].values.flatten()
    gen_vals  = gen_vals[(gen_vals >= clip[0]) & (gen_vals <= clip[1])]

    x = np.linspace(clip[0], clip[1], 500)
    ax.plot(x, stats.gaussian_kde(real_vals)(x), "b-", linewidth=2, label="Real")
    ax.plot(x, stats.gaussian_kde(gen_vals)(x), "--", color=CLASS_COLORS[cls], linewidth=2, label="Generated")
    ax.axvline(0, color="gray", linewidth=0.8, linestyle=":")
    ax.set_title(cls)
    ax.set_xlabel("Daily log-return")
    ax.set_ylabel("Density")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Pooled Return Distribution — Real vs Generated (Test Period 2024–2025)", fontsize=13)
plt.tight_layout()
plt.savefig(PCA_ROOT / "figures" / "joint_pooled_kde.png", dpi=150)
plt.show()

## 2. Summary Statistics — Per Asset Class

In [ ]:
def summary_stats(arr):
    arr = arr[~np.isnan(arr)]
    return {
        "mean":     round(arr.mean(), 6),
        "std":      round(arr.std(), 6),
        "skew":     round(float(stats.skew(arr)), 4),
        "kurtosis": round(float(stats.kurtosis(arr)), 4),
        "5th pct":  round(np.percentile(arr, 5), 6),
        "1st pct":  round(np.percentile(arr, 1), 6),
    }

rows = {}
for cls, indices in CLASSES.items():
    cols = [asset_list[i] for i in indices]
    rows[f"Real — {cls}"]      = summary_stats(real_test[cols].values.flatten())
    rows[f"Generated — {cls}"] = summary_stats(gen_df[cols].values.flatten())

summary_df = pd.DataFrame(rows).T
print(summary_df.to_string())

## 3. Correlation Matrix — Real vs Generated Per Class

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for col_idx, (cls, indices) in enumerate(CLASSES.items()):
    cols = [asset_list[i] for i in indices]

    real_corr = real_test[cols].corr()
    gen_corr  = gen_df[cols].corr()

    for row_idx, (corr, label) in enumerate([(real_corr, "Real"), (gen_corr, "Generated")]):
        ax = axes[row_idx][col_idx]
        im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
        ax.set_title(f"{label} — {cls}")
        ax.set_xticks(range(len(cols)))
        ax.set_yticks(range(len(cols)))
        ax.set_xticklabels([c.upper() for c in cols], rotation=90, fontsize=7)
        ax.set_yticklabels([c.upper() for c in cols], fontsize=7)
        plt.colorbar(im, ax=ax, fraction=0.04)

        frob = np.linalg.norm(real_corr.values - gen_corr.values, "fro")
        if row_idx == 1:
            ax.set_xlabel(f"Frobenius gap vs Real: {frob:.3f}", fontsize=9)

plt.suptitle("Correlation Matrices — Real vs Generated (Test Period)", fontsize=13)
plt.tight_layout()
plt.savefig(PCA_ROOT / "figures" / "joint_corr_matrices.png", dpi=150)
plt.show()

## 4. Cross-Class Correlation — Real vs Generated

Checks whether the generator preserves cross-asset-class dependence (e.g. bonds vs stocks during stress).

In [ ]:
# Full 60x60 correlation matrices
real_full = real_test[asset_list].corr()
gen_full  = gen_df[asset_list].corr()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (mat, title) in zip(axes[:2], [
    (real_full, "Real"),
    (gen_full,  "Generated"),
]):
    im = ax.imshow(mat.values, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
    ax.set_title(f"{title} — Full 60×60 Correlation Matrix")
    # Label class boundaries
    for boundary in [20, 40]:
        ax.axhline(boundary - 0.5, color="black", linewidth=1.5)
        ax.axvline(boundary - 0.5, color="black", linewidth=1.5)
    ax.set_xticks([]); ax.set_yticks([])
    # Class labels
    for pos, name in [(10, "Bonds"), (30, "Commodities"), (50, "Stocks")]:
        ax.text(pos, -1.5, name, ha="center", fontsize=9)
        ax.text(-1.5, pos, name, ha="right", va="center", fontsize=9, rotation=90)
    plt.colorbar(im, ax=ax, fraction=0.04)

# Difference matrix
diff = real_full.values - gen_full.values
frob_total = np.linalg.norm(diff, "fro")
im = axes[2].imshow(diff, vmin=-0.5, vmax=0.5, cmap="RdBu_r", aspect="auto")
axes[2].set_title(f"Difference (Real − Generated)\nFrobenius: {frob_total:.3f}")
for boundary in [20, 40]:
    axes[2].axhline(boundary - 0.5, color="black", linewidth=1.5)
    axes[2].axvline(boundary - 0.5, color="black", linewidth=1.5)
axes[2].set_xticks([]); axes[2].set_yticks([])
plt.colorbar(im, ax=axes[2], fraction=0.04)

plt.tight_layout()
plt.savefig(PCA_ROOT / "figures" / "joint_full_corr.png", dpi=150)
plt.show()

# Block-level Frobenius gaps
print("\nFrobenius gap by block:")
cls_names = list(CLASSES.keys())
cls_indices_list = list(CLASSES.values())
for i, (n1, idx1) in enumerate(zip(cls_names, cls_indices_list)):
    for j, (n2, idx2) in enumerate(zip(cls_names, cls_indices_list)):
        if i > j:
            continue
        block_real = real_full.values[np.ix_(idx1, idx2)]
        block_gen  = gen_full.values[np.ix_(idx1, idx2)]
        frob = np.linalg.norm(block_real - block_gen, "fro")
        print(f"  {n1} vs {n2}: {frob:.4f}")

## 5. Volatility Per Asset — Real vs Generated

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (cls, indices) in zip(axes, CLASSES.items()):
    cols = [asset_list[i] for i in indices]
    real_vol = real_test[cols].std()
    gen_vol  = gen_df[cols].std()

    x = np.arange(len(cols))
    w = 0.35
    ax.bar(x - w/2, real_vol, w, label="Real", color="steelblue", alpha=0.8)
    ax.bar(x + w/2, gen_vol,  w, label="Generated", color=CLASS_COLORS[cls], alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([c.upper() for c in cols], rotation=90, fontsize=7)
    ax.set_ylabel("Daily vol (std)")
    ax.set_title(cls)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    ratio = (gen_vol / real_vol).mean()
    ax.set_xlabel(f"Mean vol ratio gen/real: {ratio:.3f}", fontsize=9)

plt.suptitle("Per-Asset Volatility — Real vs Generated (Test Period)", fontsize=13)
plt.tight_layout()
plt.savefig(PCA_ROOT / "figures" / "joint_vol_per_asset.png", dpi=150)
plt.show()

## 6. Left Tail Exceedance — Per Asset Class

In [ ]:
thresholds = np.linspace(-0.06, -0.005, 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (cls, indices) in zip(axes, CLASSES.items()):
    cols = [asset_list[i] for i in indices]

    real_vals = real_test[cols].values.flatten()
    real_vals = real_vals[~np.isnan(real_vals)]
    gen_vals  = gen_df[cols].values.flatten()

    ax.plot(thresholds, [(real_vals < t).mean() for t in thresholds],
            "b-", linewidth=2, label="Real")
    ax.plot(thresholds, [(gen_vals < t).mean() for t in thresholds],
            "--", color=CLASS_COLORS[cls], linewidth=2, label="Generated")
    ax.set_xlabel("Return threshold")
    ax.set_ylabel("Exceedance probability")
    ax.set_title(cls)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Left Tail Exceedance — Real vs Generated per Asset Class (Test Period)", fontsize=13)
plt.tight_layout()
plt.savefig(PCA_ROOT / "figures" / "joint_tail_exceedance.png", dpi=150)
plt.show()

## 7. Pairwise Rolling Correlation — Correlated vs Uncorrelated Assets

Two pairs chosen to stress-test the generator at opposite extremes:
- **AGG / BND** (real ρ = 0.956) — two broad investment-grade bond ETFs, nearly identical assets
- **Wheat / NVDA** (real ρ = 0.025) — agricultural commodity vs semiconductor stock, essentially uncorrelated

Rolling window = 63 days (~1 quarter).

In [ ]:
WINDOW = 63  # rolling window in days (~1 quarter)

PAIRS = [
    ("wheat",     "nvda",        "Wheat vs NVDA",        "Uncorrelated — commodity vs tech"),
    ("crude_oil", "natural_gas", "Crude Oil vs Nat Gas",  "Energy — correlated but decouples sharply"),
]

n_pairs = len(PAIRS)
fig, axes = plt.subplots(n_pairs, 1, figsize=(14, 3.5 * n_pairs), sharex=False)
if n_pairs == 1:
    axes = [axes]

for ax, (a, b, title, desc) in zip(axes, PAIRS):
    roll_real = real_test[[a, b]].rolling(WINDOW).corr().unstack()[a][b].dropna()
    roll_gen  = gen_df[[a, b]].rolling(WINDOW).corr().unstack()[a][b].dropna()

    ax.plot(roll_real.index, roll_real.values, "b-",  linewidth=1.0, label="Real",      alpha=0.85)
    ax.plot(roll_gen.index,  roll_gen.values,  "r--", linewidth=1.0, label="Generated", alpha=0.85)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_title(f"{title}  [{desc}]  — real mean: {roll_real.mean():.3f}  gen mean: {roll_gen.mean():.3f}")
    ax.set_ylabel("Correlation")
    ax.set_ylim(-1, 1)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    print(f"  {title:30s}  real: {roll_real.mean():.3f}  gen: {roll_gen.mean():.3f}  gap: {abs(roll_real.mean()-roll_gen.mean()):.3f}")

axes[-1].set_xlabel("Date")
plt.suptitle(f"Rolling Pairwise Correlations (window={WINDOW}d) — PCAFactor Model [TEST SET]", fontsize=12)
plt.tight_layout()
plt.savefig(PCA_ROOT / "figures" / "joint_pairwise_rolling_corr.png", dpi=150)
plt.show()